# Datenexploration der implementierten Flutquellen

Dieses Notebook exploriert die aktuell implementierten Datenquellen aus `app/data/flood_api.py`:
- `MockFloodAdapter`
- `CopernicusEMSAdapter`
- `GloFASRapidRiskAssessmentAdapter`

Anschliessend werden die Rohdaten mit `FloodDataLoader` normalisiert, validiert und zusammengefuehrt.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from tempfile import TemporaryDirectory


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'app').exists() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Repository root nicht gefunden.')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.data.flood_api import (
    CopernicusEMSAdapter,
    GloFASRapidRiskAssessmentAdapter,
    MockFloodAdapter,
)
from app.data.loader import FloodDataLoader, InMemoryFloodStore

print(f'Repo root: {REPO_ROOT}')

In [ ]:
def polygon(min_lon: float, min_lat: float, max_lon: float, max_lat: float) -> dict:
    return {
        'type': 'Polygon',
        'coordinates': [[
            [min_lon, min_lat],
            [max_lon, min_lat],
            [max_lon, max_lat],
            [min_lon, max_lat],
            [min_lon, min_lat],
        ]],
    }


def as_rows(raw_events: list[dict]) -> list[dict]:
    rows = []
    for event in raw_events:
        rows.append({
            'event_id': event.get('event_id'),
            'source': event.get('source'),
            'observation_time': event.get('observation_time'),
            'severity': event.get('severity'),
            'confidence': event.get('confidence'),
            'geometry_type': (event.get('geometry') or {}).get('type'),
        })
    return rows


def show_table(rows: list[dict]):
    if not rows:
        print('Keine Daten')
        return

    try:
        import pandas as pd

        return pd.DataFrame(rows)
    except Exception:
        for row in rows:
            print(row)


## 1) MockFloodAdapter

In [ ]:
now = datetime.now(timezone.utc)

mock_events = [
    {
        'event_id': 'mock-a',
        'source': 'mock',
        'observation_time': (now - timedelta(hours=2)).isoformat(),
        'severity': 0.45,
        'confidence': 0.8,
        'geometry': polygon(13.2, 52.3, 13.6, 52.7),
        'properties': {'name': 'Mock Berlin A'},
    },
    {
        'event_id': 'mock-b',
        'source': 'mock',
        'observation_time': (now - timedelta(hours=30)).isoformat(),
        'severity': 0.85,
        'confidence': 0.9,
        'geometry': polygon(10.0, 45.0, 11.0, 46.0),
        'properties': {'name': 'Mock Alps B'},
    },
]

mock_adapter = MockFloodAdapter(events=mock_events)
mock_status = mock_adapter.health()
mock_recent = mock_adapter.fetch_events(
    bbox=(13.0, 52.0, 14.0, 53.0),
    since=now - timedelta(hours=24),
)

print(mock_status)
show_table(as_rows(mock_recent))

## 2) CopernicusEMSAdapter (lokales GeoJSON)

In [ ]:
with TemporaryDirectory() as tmp_dir:
    geojson_path = Path(tmp_dir) / 'copernicus_sample.geojson'
    payload = {
        'type': 'FeatureCollection',
        'features': [
            {
                'id': 'cop-1',
                'type': 'Feature',
                'geometry': polygon(12.2, 50.0, 13.1, 50.9),
                'properties': {
                    'observation_time': (now - timedelta(hours=1)).isoformat(),
                    'severity': 0.75,
                    'confidence': 0.88,
                    'name': 'Copernicus Zone 1',
                },
            },
            {
                'id': 'cop-old',
                'type': 'Feature',
                'geometry': polygon(8.0, 47.0, 8.5, 47.5),
                'properties': {
                    'observation_time': (now - timedelta(days=10)).isoformat(),
                    'severity': 0.2,
                    'confidence': 0.5,
                },
            },
        ],
    }
    geojson_path.write_text(json.dumps(payload), encoding='utf-8')

    cop_adapter = CopernicusEMSAdapter(str(geojson_path))
    cop_status = cop_adapter.health()
    cop_events = cop_adapter.fetch_events(
        bbox=(12.0, 49.5, 13.5, 51.2),
        since=now - timedelta(days=7),
    )

print(cop_status)
show_table(as_rows(cop_events))

## 3) GloFASRapidRiskAssessmentAdapter (simulierte API-Responses)

Hinweis: Fuer reproduzierbare Exploration ohne Netzwerkanbindung wird `_request_json` in einer Unterklasse ueberschrieben.

In [ ]:
class FakeGloFASAdapter(GloFASRapidRiskAssessmentAdapter):
    def __init__(self, payload: dict):
        super().__init__(base_url='https://example.test/api', api_key='demo')
        self._payload = payload

    def _request_json(self, url: str) -> dict:
        return self._payload


feature_collection_payload = {
    'type': 'FeatureCollection',
    'features': [
        {
            'id': 'glofas-fc-1',
            'type': 'Feature',
            'geometry': polygon(9.8, 44.8, 11.2, 46.2),
            'properties': {
                'observation_time': (now - timedelta(hours=4)).isoformat(),
                'severity': 0.82,
                'confidence': 0.91,
            },
        }
    ],
}

results_payload = {
    'results': [
        {
            'event_id': 'glofas-r-1',
            'bbox': [12.0, 50.0, 13.0, 51.0],
            'timestamp': (now - timedelta(hours=3)).isoformat(),
            'risk_score': 0.66,
            'probability': 0.73,
        }
    ]
}

glofas_fc_adapter = FakeGloFASAdapter(feature_collection_payload)
glofas_results_adapter = FakeGloFASAdapter(results_payload)

glofas_fc_events = glofas_fc_adapter.fetch_events(
    bbox=(9.0, 44.0, 12.0, 47.0),
    since=now - timedelta(hours=24),
)
glofas_result_events = glofas_results_adapter.fetch_events(
    bbox=(11.5, 49.5, 13.2, 51.2),
    since=now - timedelta(hours=24),
)

print('FeatureCollection-Variante:')
show_table(as_rows(glofas_fc_events))

print('\nResults-Variante:')
show_table(as_rows(glofas_result_events))

## 4) Normalisierung, Validierung, Merge und Store

In [ ]:
all_raw = []
all_raw.extend(mock_recent)
all_raw.extend(cop_events)
all_raw.extend(glofas_fc_events)
all_raw.extend(glofas_result_events)

loader = FloodDataLoader()
normalized = [loader.normalize(raw) for raw in all_raw]
valid = [event for event in normalized if loader.validate(event)]
merged = loader.merge(valid)

store = InMemoryFloodStore()
store.upsert_events(merged)

print(f'Raw: {len(all_raw)} | Normalized: {len(normalized)} | Valid: {len(valid)} | Merged: {len(merged)}')

summary_rows = [
    {
        'event_id': e.event_id,
        'source': e.source,
        'observation_time': e.observation_time.isoformat(),
        'severity': e.severity,
        'confidence': e.confidence,
    }
    for e in merged
]
show_table(summary_rows)

In [ ]:
query_bbox = (12.0, 49.5, 14.0, 53.0)
at_time = datetime.now(timezone.utc)

bbox_hits = store.query_bbox(query_bbox)
active_hits = store.query_active(at_time=at_time, max_age_hours=24)

print(f'query_bbox Treffer: {len(bbox_hits)}')
print(f'query_active (24h) Treffer: {len(active_hits)}')

show_table([
    {
        'event_id': e.event_id,
        'source': e.source,
        'age_min': e.data_age_minutes(at_time),
        'severity': e.severity,
        'confidence': e.confidence,
    }
    for e in active_hits
])

## Naechste Erweiterungen

- Echte Copernicus-GeoJSON-Datei statt synthetischer Daten einbinden
- `GloFASRapidRiskAssessmentAdapter.health()` gegen die reale API pruefen
- Explorative Visualisierung (z. B. Histogramm von `severity` pro `source`)